## Load from env

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME")
API_VERSION = os.getenv("API_VERSION")

## Initialize environment variables for OpenAI

In [ ]:
import os
AZURE_OPENAI_KEY = AZURE_OPENAI_KEY
AZURE_OPENAI_ENDPOINT = AZURE_OPENAI_ENDPOINT
DEPLOYMENT_NAME = DEPLOYMENT_NAME
API_VERSION = API_VERSION

os.environ["AZURE_OPENAI_KEY"] = AZURE_OPENAI_KEY
os.environ["AZURE_OPENAI_API_BASE"] = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_OPENAI_API_VERSION"] = API_VERSION
os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = "gpt-4.1"

## LLM CALL

In [ ]:
import json
from langchain_openai import AzureChatOpenAI
from langchain_core import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    api_key=AZURE_OPENAI_KEY,
    max_retries=2,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a summarizer agent. Your aim is to summarize the input text clearly and concisely."),
    ("human","{input text}")
])

simple_summarizer_agent = prompt | llm | StrOutputParser()

In [ ]:
text = '''
Artificial Intelligence (AI) is the ability of computers or machines to mimic human intelligence, such as learning from experience, recognizing patterns, making decisions, and solving problems. Instead of being explicitly programmed for every action, AI systems analyze data to perform tasks independently, such as identifying objects in photos, translating languages, or offering tailored recommendations.
'''

In [ ]:
result = simple_summarizer_agent.invoke({"input text": text})
print(result)

## Memory Saver

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    api_key=AZURE_OPENAI_KEY,
    max_retries=2,
)

@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location."""
    weather_data = {
        "new york": "Sunny, 25°C",
        "san francisco": "Foggy, 15°C",
        "london": "Rainy, 10°C"
    }
    return weather_data.get(location.lower(), f"Weather data not available for {location}.")

@tool
def calculate_sum(a: int, b: int) -> str:
    """Calculate the sum of two numbers."""
    return f"The sum of {a} and {b} is {a + b}."

tools = [get_current_weather, calculate_sum]
print(f"Tools registered: {[tool.name for tool in tools]}")

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = """
You are a helpful assistant with access to the following tools:
- get_current_weather: checks the current weather in a given location.
- calculate_sum: calculates the sum of two numbers.
Think step by step and use the tools when necessary to answer the user's question.
"""

memory = MemorySaver()

react_agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = SYSTEM_PROMPT,
    checkpoint=memory
)

print("Agent compiled successfully with session memory")

In [ ]:
from langchain_core.messages import HumanMessage
import uuid

def run_agent(user_input:str, session_id:str) -> str:
    """
    Invoke the agent for a given session.
    Each unique session_id gets its own isolated conversation history.
    Pass the same seesion_id across turns to continue a conversation.
    """
    config = {"configurable" : {"thread_id":session_id}}
    result = react_agent.invoke(
        {"message":[HumanMessage(content=user_input)]},
        config = config
    )
    return result["messages"][-1].content

In [ ]:
# share the same session_id across turns to continue the conversation
SESSION_B = str(uuid.uuid4())
print(f"Session ID: {SESSION_B}")

turn1 = "What's the weather like in New York and what's 5 plus 7?"
print(f"Turn 1 - User: {turn1}")
print(f"Turn 1 - Agent: {run_agent(turn1, SESSION_B)}")

turn2 = "what did i just ask you to calculate?"
print(f"Turn 2 - User: {turn2}")
print(f"Turn 2 - Agent: {run_agent(turn2, SESSION_B)}")

## Simple Agent

In [ ]:
import json
from langgraph.graph import StateGraph
from typing import TypedDict

class SummarizerState(TypedDict):
    input_text: str
    summary: str

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    api_key=AZURE_OPENAI_KEY,
    max_retries=2,
)

def summarizer_node(state: SummarizerState) -> SummarizerState:
    messages = [
        {"role": "system", "content": "You are a helpful assistant that summarizes text."},
        {"role": "user", "content": state['input_text']}
    ]
    response = llm.invoke(messages)
    return {"summary":response.content}

graph = StateGraph(SummarizerState)
graph.add_node("summarizer", summarizer_node)
graph.set_entry_point("summarizer")
graph.set_finish_point("summarizer")

simple_summarizer_agent = graph.compile()

In [ ]:
text = '''
Artificial Intelligence (AI) is the ability of computers or machines to mimic human intelligence, such as learning from experience, recognizing patterns, making decisions, and solving problems. Instead of being explicitly programmed for every action, AI systems analyze data to perform tasks independently, such as identifying objects in photos, translating languages, or offering tailored recommendations.
'''

result = simple_summarizer_agent.invoke({"input_text": text})
print(result["summary"])

## Agent Executor

In [ ]:
text = '''
Artificial Intelligence (AI) is the ability of computers or machines to mimic human intelligence, such as learning from experience, recognizing patterns, making decisions, and solving problems. Instead of being explicitly programmed for every action, AI systems analyze data to perform tasks independently, such as identifying objects in photos, translating languages, or offering tailored recommendations.
'''

import json
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromotTemoplate, MessagesPlaceholder

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    api_key=AZURE_OPENAI_KEY,
    max_retries=2,
)

@tool
def format_summary(text: str) -> str:
    """Format the summary in a concise manner."""
    return f"Summary: {text.strip()}"

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that summarizes text."),
    ("human", "{input_text}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

tools = [format_summary]
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
simple_summarizer_agent = AgentExecutor(name = "SimpleSummarizer", agent = agent, tools=tools, verbose=True)

result = simple_summarizer_agent.invoke({"input:": "Your long text to summarize..." + text})
print(result["output"])

## Orchestrator Agent with multipart task detection, sequential routing and pending task tracking for complex queries that require multiple agents in sequence

In [ ]:
import json
import re
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph,END
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool
from typing import TypedDict, List, Literal
from langchain_core.messages import HumanMessage,AIMessage,BaseMessage

In [ ]:
class MultiAgentState(TypedDict):
    messages : List[BaseMessage]
    next_agent: str
    task_type: str
    remaining_subtasks: List[str]
    remaining_task_types: List[str]

In [ ]:
@tool
def summarize_text(text:str)->str:
    """summarize text using AzureChatOpenAI"""
    llm = AzureChatOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        azure_deployment=DEPLOYMENT_NAME,
        api_version=API_VERSION,
        api_key=AZURE_OPENAI_KEY,
        max_retries=2,
    )
    response = [
        {"role": "system", "content": (
            "You are a text summarization expert. Your ONLY task is to summarize provided text"
            "clearly and concisely. Do not provide any additional information or commentary beyond the summary."
        )},
        {"role": "user", "content": f"Summarize the following text: {text}"}
    ]
    return response.content if hasattr(response,"content") else str(response)

@tool
def solve_math(question: str) -> str:
    """solve math problem using AzureChatOpenAI"""
    llm = AzureChatOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        azure_deployment=DEPLOYMENT_NAME,
        api_version=API_VERSION,
        api_key=AZURE_OPENAI_KEY,
        max_retries=2,
    )
    response = [
        {"role": "system", "content": "You are a helpful assistant that solves math problems."},
        {"role": "user", "content": question}
    ]
    return response.content if hasattr(response,"content") else str(response)

@tool
def research_topic(topic: str) -> str:
    """research a topic using AzureChatOpenAI"""
    llm = AzureChatOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        azure_deployment=DEPLOYMENT_NAME,
        api_version=API_VERSION,
        api_key=AZURE_OPENAI_KEY,
        max_retries=2,
    )
    response = [
        {"role": "system", "content": (
            "You are a research assistant. Your ONLY task is to provide detailed, factual information"
            "and explanations on topics. Do NOT write creative content, summarize text, solve math,"
            "or provide opinions. Focus solely on delivering accurate and comprehensive information based on the topic provided."
        )},
        {"role": "user", "content": f"Research the following topic and provide a concise summary: {topic}"}
    ]
    return response.content if hasattr(response,"content") else str(response)

In [ ]:
summarize_llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    api_key=AZURE_OPENAI_KEY,
    max_retries=2,
)

math_llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    api_key=AZURE_OPENAI_KEY,
    max_retries=2,
)

research_llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    api_key=AZURE_OPENAI_KEY,
    max_retries=2,
)

summarize_llm = create_react_agent(summarize_llm,[summarize_text])
math_llm = create_react_agent(math_llm,[solve_math])
research_llm = create_react_agent(research_llm,[research_topic])

orchestrator_llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    api_key=AZURE_OPENAI_KEY,
    max_retries=2,
)

In [ ]:
def _extract_json(raw:str)->dict | None:
    """Robustly extract a JSON object from an LLM response.
    Handles markdown code fences and plain JSON text.
    """
    text = re.sub(r"```(?:json)?\s*", "", raw).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return None

In [ ]:
def orchestrator_node(state:MultiAgentState):
    """Enhanced orchestrator that detects MULTI-PART queries and routes them sequentially"""
    messages = state["messages"]
    user_query = messages[-1].content 

    # First detect if the query contains multiple distinct tasks (e.g. "Summarize this text and also solve this math problem")
    detection_instruction = (
        "Analyze this user query and identify if it contains MULTIPLE distinct tasks." 
        "Return JSON with: 'is_multi_part': true/false, 'subtasks':[list of individual tasks],'task_types':[list of task types]\n"
        "Example: 'Calculate 15*8+12, then research math history, then summarize' -> "
        "{'is_multi_part': true, 'subtasks':['Calculate 15*8+12','research math history','summarize'], 'task_types':['math','research','summarization']}"
    )
    
    detection_messages =[
        {"role":"system","content":detection_instruction},
        {"role":"user","content":f"Query: {user_query}"}
    ]

    detection_resp = orchestrator_llm.invoke(detection_messages)
    detection_raw = getattr(detection_resp,"content",str(detection_resp))
    detection = _extract_json(detection_raw)

    print(f"Detection output: {detection}")

    # if multi-part, route
    if detection and detection.get("is_multi_part"):
        subtasks = detection.get("subtasks",[])
        task_types = detection.get("task_types",[])

        if subtasks and task_types:
            first_subtask = subtasks[0]
            first_task_type = task_types[0].lower()
            remaining_subtasks = subtasks[1:]

            # store remaining subtasks in state for sequential routing
            rational_message = AIMessage(
                content=f"[Orchestrator] Detected multi-part query. Processing subtask 1 of {len(subtasks)}:'{first_subtask}'"
            )

            valid_map = {
                "summarization": "summarize agent",
                "math": "math agent",
                "research": "research agent"
            }

            next_agent = valid_map.get(first_task_type, "end")

            return {
                "next_agent": next_agent,
                "task_type": first_task_type,
                "messages" : messages + [
                    HumanMessage(content=first_subtask),
                    rational_message
                ],
                # store remaining subtasks for next routing decision
                "remaining_subtasks": remaining_subtasks,
                "remaining_task_types": task_types[1:],
            }
    
    # Original single-task routing logic (fallback)
    decision_instruction = (
        "You are an autonomous orchestrator deciding which specialized agent sould handle the user query.\n"
        "Available agents and their STRICT domains:\n"
        "- summarize agent: ONLY summarizes text\n"
        "- math agent: ONLY solves math problems\n"
        "- research agent: ONLY answers research questions\n"
    )

    prompt_messages = [
        {"role":"system","content":decision_instruction},
        {"role":"user","content":f"User query:\n{user_query}"}
    ]

    llm_resp = orchestrator_llm.invoke(prompt_messages)
    raw_content=getattr(llm_resp,"content",str(llm_resp))
    print("Raw LLM response:", raw_content)

    decision = _extract_json(raw_content)
    print("Parsed decision:", decision)

    valid_map = {
                "summarization": "summarize agent",
                "math": "math agent",
                "research": "research agent"
            }
    
    if decision is None:
        print("WARNING: JSON parsing failed, defaulting to none.")
        task_type = "none"
    else:
        task_type = decision.get("task_type","").lower()

    if task_type not in valid_map:
        error_message = AIMessage(content=(
            f"[Orchestrator] ERROR: Unrecognized or missing task type in LLM response. Received task_type: '{task_type}'."
            "Defaulting to 'none' which will end the conversation."
        ))
        return {
            "next_agent" : "end",
            "task_type": task_type,
            "messages": messages + [error_message],
        }
    
    next_agent = valid_map[task_type]
    rational = decision.get("rationale","No rationale provided.")
    rational_message = AIMessage(content=f"[Orchestrator] Routing to {next_agent} based on detected task type '{task_type}'. Rationale: {rational}")

    return {
        "next_agent": next_agent,
        "task_type": task_type,
        "messages": messages + [rational_message],
        "remaining_subtasks": [],
        "remaining_task_types": []
    }


In [ ]:
def summarizer_node(state:MultiAgentState):
    if state.get("task_type") != "summarization":
        rejection = AIMessage(content="[Summarizer] ERROR: Received task type does not match summarization. Ending conversation.")
        return {
            "messages": state["messages"] + [rejection],
            "next_agent": "end",
            "task_type": state["messages"]
        }
    messages = state["messages"]
    result = summarizer_agent.invoke({"messages": messages})
    # check if there are more subtasks
    remaining_subtasks = state.get("remaining_subtasks",[])
    if remaining_subtasks:
        #route to next subtask
        next_task_type = state.get("remaining_task_types",[""][0].lower())
        valid_map = {
            "summarization": "summarize agent",
            "math": "math agent",
            "research": "research agent"
        }
        next_agent = valid_map.get(next_task_type,"end")
        return {
            "messages" : result["messages"] + [HumanMessage(content=remaining_subtasks[0])],
            "next_agent": next_agent,
            "task_type": next_task_type,
            "remaining_subtasks": remaining_subtasks[1:],
            "remaining_task_types": state.get("remaining_task_types",[])[1:]
         }
    return {
        "messages": result["messages"],
        "next_agent": "end",
        "task_type": state["task_type"],
        "remaining_subtasks": [],
        "remaining_task_types": []
    }


def math_node(state:MultiAgentState):
    if state.get("task_type") != "math":
        rejection = AIMessage(content="[Math Agent] ERROR: Received task type does not match math. Ending conversation.")
        return {
            "messages": state["messages"] + [rejection],
            "next_agent": "end",
            "task_type": state["messages"]
        }
    messages = state["messages"]
    result = math_agent.invoke({"messages": messages})
    # check if there are more subtasks
    remaining_subtasks = state.get("remaining_subtasks",[])
    if remaining_subtasks:
        #route to next subtask
        next_task_type = state.get("remaining_task_types",[""][0].lower())
        valid_map = {
            "summarization": "summarize agent",
            "math": "math agent",
            "research": "research agent"
        }
        next_agent = valid_map.get(next_task_type,"end")
        return {
            "messages" : result["messages"] + [HumanMessage(content=remaining_subtasks[0])],
            "next_agent": next_agent,
            "task_type": next_task_type,
            "remaining_subtasks": remaining_subtasks[1:],
            "remaining_task_types": state.get("remaining_task_types",[])[1:]
         }
    return {
        "messages": result["messages"],
        "next_agent": "end",
        "task_type": state["task_type"],
        "remaining_subtasks": [],
        "remaining_task_types": []
    }

def research_node(state:MultiAgentState):
    if state.get("task_type") != "research":
        rejection = AIMessage(content="[Research Agent] ERROR: Received task type does not match research. Ending conversation.")
        return {
            "messages": state["messages"] + [rejection],
            "next_agent": "end",
            "task_type": state["messages"]
        }
    messages = state["messages"]
    result = research_agent.invoke({"messages": messages})
    # check if there are more subtasks
    remaining_subtasks = state.get("remaining_subtasks",[])
    if remaining_subtasks:
        #route to next subtask
        next_task_type = state.get("remaining_task_types",[""][0].lower())
        valid_map = {
            "summarization": "summarize agent",
            "math": "math agent",
            "research": "research agent"
        }
        next_agent = valid_map.get(next_task_type,"end")
        return {
            "messages" : result["messages"] + [HumanMessage(content=remaining_subtasks[0])],
            "next_agent": next_agent,
            "task_type": next_task_type,
            "remaining_subtasks": remaining_subtasks[1:],
            "remaining_task_types": state.get("remaining_task_types",[])[1:]
         }
    return {
        "messages": result["messages"],
        "next_agent": "end",
        "task_type": state["task_type"],
        "remaining_subtasks": [],
        "remaining_task_types": []
    }

In [ ]:
def route_to_agent(state: MultiAgentState) -> Literal["summarize agent","math agent","research agent","end"]:
    next_agent = state.get("next_agent","end")
    # if next_agent not in ["summarize agent","math agent","research agent"]:
    #     return "end"
    return next_agent

In [ ]:
workflow = StateGraph(MultiAgentState)
workflow.add_node("orchestrator", orchestrator_node)
workflow.add_node("summarize agent", summarizer_node)
workflow.add_node("math agent", math_node)
workflow.add_node("research agent", research_node)

workflow.set_entry_point("orchestrator")

workflow.add_conditional_edge(
    "orchestrator",
    route_to_agent,
    {
        "summarize agent": "summarize agent",
        "math agent": "math agent",
        "research agent": "research agent",
        "end": END
    }
)

workflow.add_conditional_edge(
    "summarize agent",
    route_to_agent,
    {
        "summarize agent": "summarize agent",
        "math agent": "math agent",
        "research agent": "research agent",
        "end": END
    }
)

workflow.add_conditional_edge(
    "math agent",
    route_to_agent,
    {
        "summarize agent": "summarize agent",
        "math agent": "math agent",
        "research agent": "research agent",
        "end": END
    }
)

workflow.add_conditional_edge(
    "research agent",
    route_to_agent,
    {
        "summarize agent": "summarize agent",
        "math agent": "math agent",
        "research agent": "research agent",
        "end": END
    }
)

In [ ]:
multi_agent_system = workflow.compile()
print("Multi-agent system compiled successfully.")

In [ ]:
query = "Calculate 15*8+12, then research the history of the mathematics, then summarize the key developments in 2 sentences."

result = multi_agent_system.invoke({
    "messages": [HumanMessage(content=query)],
    "next_agent": "",
    "task_type": ""
})

print("Final output:", result)